# Tennessee River data atlas
Read-only exploration of downloaded geography, NWM, AORC, CAMELS-H and TVA data. Run on your GPU server. Missing or still-downloading datasets are skipped. Maps use simplified geometry for display only; source files are unchanged.

Install plotting dependencies in the download environment before launching:
```bash
conda activate /data/tva_data/conda_envs/tva_download
python -m pip install --no-cache-dir jupyterlab folium matplotlib ipywidgets
jupyter lab --no-browser --ip=127.0.0.1 --port=8888
```
From your laptop, forward the port using `ssh -L 8888:localhost:8888 talhamuh@YOUR_SERVER`, then open the token URL printed by Jupyter. Keep Jupyter bound to localhost.


In [ ]:
from pathlib import Path
import json, zipfile, io, html
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import folium
from folium.plugins import MarkerCluster
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path('/data/tva_data')
# Set either a ZIP path or an extracted TVA_headwater directory.
TVA_PATH = ROOT / 'raw/tva/TVA_headwater.zip'
SIMPLIFY_METERS = 10  # display only; all reaches retained
EXCLUDED_TVA_IDS = {'BRRT1', 'PORT1S', 'CDZK2', 'EMGT1', 'CAFT1', 'MCGT1'}


## Local file inventory
Counts describe files on disk, not verified temporal completeness. Partial files are excluded.

In [ ]:
patterns = {'NWM': 'raw/nwm/*.nc', 'AORC': 'raw/aorc/**/*.nc',
            'CAMELS-H': 'raw/camelsh/flow/*.nc'}
files = {k: sorted(ROOT.glob(v)) for k,v in patterns.items()}
display(pd.DataFrame([{'dataset':k, 'completed_nc_files':len(v),
                      'disk_GiB':round(sum(p.stat().st_size for p in v)/2**30,3)}
                     for k,v in files.items()]))
print('TVA available:', TVA_PATH.exists())


In [ ]:
def optional_geo(path):
    if not path.exists():
        print('Not available:', path)
        return gpd.GeoDataFrame(geometry=[], crs=4326)
    return (gpd.read_parquet(path) if path.suffix == '.parquet' else gpd.read_file(path)).to_crs(4326)

boundary = optional_geo(ROOT/'raw/geography/tennessee_huc2.geojson')
huc8 = optional_geo(ROOT/'raw/geography/tennessee_huc8.geojson')
reaches = optional_geo(ROOT/'metadata/tennessee_flowlines.parquet')
gauge_path = ROOT/'raw/camelsh/metadata/tennessee_camelsh_gauges.csv'
gauges = pd.read_csv(gauge_path, dtype={'STAID':str}) if gauge_path.exists() else pd.DataFrame()
if not gauges.empty:
    gauges['STAID'] = gauges.STAID.str.zfill(8)
    gauges['downloaded'] = gauges.STAID.map(lambda s:(ROOT/'raw/camelsh/flow'/f'{s}_hourly.nc').exists())

def tva_read(relative):
    if TVA_PATH.is_dir(): return (TVA_PATH/relative).read_bytes()
    with zipfile.ZipFile(TVA_PATH) as z:
        names=[n for n in z.namelist() if not n.startswith('__MACOSX/') and n.endswith('/'+relative)]
        if len(names)!=1: raise ValueError(f'Ambiguous or missing TVA member: {relative}')
        return z.read(names[0])

tva = pd.read_csv(io.BytesIO(tva_read('TVA_headwater_basin_CAMELSH_StreamCat_attributes.csv')), dtype={'camelsh_usgs_id': str}) if TVA_PATH.exists() else pd.DataFrame()
print('Flowlines:',len(reaches),'CAMELS locations:',len(gauges),'TVA outlets:',len(tva))

from pyproj import Geod
pairs = tva.copy()
if not pairs.empty:
    pairs['excluded'] = pairs.tva_nwsid.isin(EXCLUDED_TVA_IDS)
    pair_columns = ['tva_station_latitude','tva_station_longitude',
                    'camelsh_station_latitude','camelsh_station_longitude']
    pairs['has_paired_coordinates'] = pairs[pair_columns].notna().all(axis=1)
    pairs['separation_m'] = np.nan
    good = pairs.has_paired_coordinates
    if good.any():
        _, _, distances = Geod(ellps='WGS84').inv(
            pairs.loc[good,'tva_station_longitude'].to_numpy(),
            pairs.loc[good,'tva_station_latitude'].to_numpy(),
            pairs.loc[good,'camelsh_station_longitude'].to_numpy(),
            pairs.loc[good,'camelsh_station_latitude'].to_numpy())
        pairs.loc[good,'separation_m'] = distances
    pairs['camelsh_flow_downloaded'] = pairs.camelsh_usgs_id.map(
        lambda sid: pd.notna(sid) and (ROOT/'raw/camelsh/flow'/f'{sid}_hourly.nc').exists())
    tva_selected = tva.loc[~tva.tva_nwsid.isin(EXCLUDED_TVA_IDS)].copy()
    display(pairs[['tva_nwsid','camelsh_usgs_id','camelsh_match_basis',
                   'separation_m','camelsh_flow_downloaded','excluded']])
else:
    tva_selected = tva.copy()
    print('TVA not loaded: set TVA_PATH to the actual ZIP or extracted directory.')


## Interactive overview
Toggle layers in the top-right corner. Gray river lines are a deterministic display sample; gauge markers show actual downloaded-file availability. Basemap tiles require internet; overlays still work without tiles.

In [ ]:
def base_map():
    m = folium.Map(location=[35.7,-85.0], zoom_start=7,
                   tiles='OpenStreetMap', prefer_canvas=True)
    if not boundary.empty:
        b = boundary.to_crs(4326)
        xmin,ymin,xmax,ymax = b.total_bounds
        m.fit_bounds([[ymin,xmin],[ymax,xmax]])
        folium.GeoJson(b.to_json(),name='Tennessee boundary',
            style_function=lambda f: {'color':'black','weight':2,'fillOpacity':0}).add_to(m)
    return m

def add_lines(m, frame, name, color, weight=0.8, show=True):
    if frame.empty: return
    cols = ['comid','geometry'] if 'comid' in frame else ['geometry']
    g = frame[cols].copy()
    g = g.loc[g.geometry.notna() & ~g.geometry.is_empty].to_crs(5070)
    g.geometry = g.geometry.simplify(SIMPLIFY_METERS, preserve_topology=True)
    tooltip = folium.GeoJsonTooltip(fields=['comid'],aliases=['COMID:']) if 'comid' in g else None
    folium.GeoJson(g.to_crs(4326).to_json(),name=name,show=show,smooth_factor=0,
        tooltip=tooltip,style_function=lambda f: {'color':color,'weight':weight,'opacity':0.8}).add_to(m)

m = base_map()
if not huc8.empty:
    folium.GeoJson(huc8.to_crs(4326).to_json(),name='HUC8 boundaries',show=False,
        style_function=lambda f: {'color':'#7251a1','weight':1,'fillOpacity':0}).add_to(m)
add_lines(m,reaches,f'Complete river network ({len(reaches):,} reaches)','#648cba')
order_col = next((c for c in reaches.columns if c.lower() in {'streamorde','streamorder','stream_order'}),None)
if order_col:
    add_lines(m,reaches.loc[pd.to_numeric(reaches[order_col],errors='coerce')>=4],
              'Larger rivers','#175a91',2)
if 'in_nwm_v30' in reaches:
    add_lines(m,reaches.loc[reaches.in_nwm_v30.eq(False)],'Unmatched NWM reaches','#e63946',2,False)
if 'has_catchment' in reaches:
    add_lines(m,reaches.loc[reaches.has_catchment.eq(False)],'Reaches without catchments','#f4a261',2,False)

regional = folium.FeatureGroup(name='All regional CAMELS-H gauges',show=False).add_to(m)
if not gauges.empty:
    for _,r in gauges.dropna(subset=['LAT_GAGE','LNG_GAGE']).iterrows():
        folium.CircleMarker([r.LAT_GAGE,r.LNG_GAGE],radius=4,
            color='green' if bool(r.downloaded) else 'gray',fill=True,
            tooltip=f'USGS {r.STAID}: '+('downloaded' if bool(r.downloaded) else 'no local flow file')).add_to(regional)

included = folium.FeatureGroup(name='Included TVA outlets').add_to(m)
excluded = folium.FeatureGroup(name='Excluded TVA outlets (outside HUC2 06)',show=False).add_to(m)
matched = folium.FeatureGroup(name='CAMELS-H locations paired to included TVA outlets').add_to(m)
links = folium.FeatureGroup(name='TVA–CAMELS-H correspondence lines',show=False).add_to(m)
if not pairs.empty:
    for _,r in pairs.iterrows():
        if pd.isna(r.tva_station_latitude) or pd.isna(r.tva_station_longitude): continue
        sid = html.escape(str(r.tva_nwsid))
        cid = html.escape(str(r.camelsh_usgs_id)) if pd.notna(r.camelsh_usgs_id) else 'None'
        distance = f'{r.separation_m:.1f} m' if pd.notna(r.separation_m) else 'Unavailable'
        content = (f'<b>TVA {sid}</b><br>USGS/CAMELS-H ID: {cid}'
            f'<br>Coordinate separation: {distance}'
            f'<br>Match basis: {html.escape(str(r.camelsh_match_basis))}'
            f'<br>Local CAMELS-H flow file: {bool(r.camelsh_flow_downloaded)}'
            f'<br>Experiment: {"Excluded" if r.excluded else "Included"}')
        tloc = [float(r.tva_station_latitude),float(r.tva_station_longitude)]
        folium.Marker(tloc,tooltip=f'TVA {sid} ↔ USGS {cid}',
            popup=folium.Popup(content,max_width=420),
            icon=folium.Icon(color='orange' if r.excluded else 'purple',icon='info-sign')).add_to(excluded if r.excluded else included)
        if r.has_paired_coordinates and not r.excluded:
            cloc = [float(r.camelsh_station_latitude),float(r.camelsh_station_longitude)]
            folium.CircleMarker(cloc,radius=9,color='#168a45',weight=3,fill=False,
                tooltip=f'USGS {cid} ↔ TVA {sid}',popup=folium.Popup(content,max_width=420)).add_to(matched)
            folium.PolyLine([tloc,cloc],color='#e67e22',weight=3,dash_array='5,5',
                tooltip=f'{sid} ↔ {cid}: {distance} (metadata link)').add_to(links)
folium.LayerControl(collapsed=False).add_to(m)
display(m)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Project all layers consistently; retain every river reach.
rivers = reaches.to_crs(5070)
basin = boundary.to_crs(5070)

fig, ax = plt.subplots(figsize=(18, 12), dpi=150)

# Complete network — no sampling or geometry simplification.
rivers.plot(ax=ax, color="#73a9cf", linewidth=0.35, zorder=1)

# Emphasize larger rivers if stream-order metadata exists.
order_col = next(
    (c for c in rivers.columns if c.lower() in
     {"streamorde", "streamorder", "stream_order"}),
    None
)
if order_col:
    order = pd.to_numeric(rivers[order_col], errors="coerce")
    major = rivers.loc[order >= 4]
    if not major.empty:
        major.plot(ax=ax, color="#175a91", linewidth=1.1, zorder=2)

if not basin.empty:
    basin.boundary.plot(ax=ax, color="#333333", linewidth=1.3, zorder=3)

legend = [Line2D([0], [0], color="#73a9cf", lw=2,
                 label=f"Full river network ({len(rivers):,} reaches)")]

# CAMELS-H gauge locations.
if not gauges.empty:
    valid = gauges.dropna(subset=["LNG_GAGE", "LAT_GAGE"]).copy()
    points = gpd.GeoDataFrame(
        valid,
        geometry=gpd.points_from_xy(valid.LNG_GAGE, valid.LAT_GAGE),
        crs=4326
    ).to_crs(5070)

    downloaded = points["downloaded"].fillna(False).astype(bool)
    for mask, color, label in [
        (downloaded, "#239b56", "CAMELS-H: flow downloaded"),
        (~downloaded, "#888888", "CAMELS-H: no local flow file")
    ]:
        if mask.any():
            points.loc[mask].plot(
                ax=ax, color=color, markersize=25,
                edgecolor="white", linewidth=0.4, zorder=4
            )
            legend.append(Line2D(
                [0], [0], marker="o", color="none",
                markerfacecolor=color, markersize=7, label=label
            ))

# TVA outlets and station IDs.
if not tva.empty:
    valid = tva.dropna(
        subset=["tva_station_longitude", "tva_station_latitude"]
    ).copy()
    outlets = gpd.GeoDataFrame(
        valid,
        geometry=gpd.points_from_xy(
            valid.tva_station_longitude, valid.tva_station_latitude
        ),
        crs=4326
    ).to_crs(5070)

    outlets.plot(
        ax=ax, marker="^", color="#a020a0", markersize=65,
        edgecolor="white", linewidth=0.6, zorder=5
    )
    for _, row in outlets.iterrows():
        ax.annotate(
            str(row.tva_nwsid),
            (row.geometry.x, row.geometry.y),
            xytext=(4, 4), textcoords="offset points",
            fontsize=7, color="#702070", zorder=6
        )
    legend.append(Line2D(
        [0], [0], marker="^", color="none",
        markerfacecolor="#a020a0", markersize=9, label="TVA outlet"
    ))

if not basin.empty:
    xmin, ymin, xmax, ymax = basin.total_bounds
    margin = max(xmax - xmin, ymax - ymin) * 0.025
    ax.set_xlim(xmin - margin, xmax + margin)
    ax.set_ylim(ymin - margin, ymax + margin)

ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title("Tennessee River basin — complete river network and gauges",
             fontsize=17)
ax.legend(handles=legend, loc="lower left", framealpha=0.95)
plt.tight_layout()
plt.show()

## Catchment detail
Choose a bounding box (west, south, east, north). This reads only intersecting shapefile features and shows up to 2,000 catchments to keep the browser responsive. The map explicitly reports truncation.

In [ ]:
CATCHMENT_BBOX = (-83.0,35.3,-82.6,35.7)
manifest_path=ROOT/'metadata/nhdplus_download.json'
if manifest_path.exists():
    manifest=json.loads(manifest_path.read_text())
    component=next((c for c in manifest['components'] if c['component']=='NHDPlusCatchment'),None)
    candidates=list(Path(component['extracted_directory']).rglob('Catchment.shp')) if component else []
    if len(candidates)==1:
        import pyogrio
        from pyproj import Transformer
        info=pyogrio.read_info(candidates[0])
        bbox=Transformer.from_crs(4326,info['crs'],always_xy=True).transform_bounds(*CATCHMENT_BBOX)
        catch=gpd.read_file(candidates[0],bbox=bbox).to_crs(4326)
        print('Intersecting catchments:',len(catch),'Displayed:',min(len(catch),2000))
        cm=base_map()
        if not catch.empty:
            c=catch.iloc[:2000].copy().to_crs(5070);c.geometry=c.geometry.simplify(30);c=c.to_crs(4326)
            folium.GeoJson(c.to_json(),name='Incremental catchments',style_function=lambda f:{'color':'#287271','weight':1,'fillOpacity':0.12}).add_to(cm)
            cm.fit_bounds([[CATCHMENT_BBOX[1],CATCHMENT_BBOX[0]],[CATCHMENT_BBOX[3],CATCHMENT_BBOX[2]]])
            display(cm)
    else: print('Missing or ambiguous Catchment.shp:',len(candidates))
else: print('NHDPlus manifest not available yet.')


## NWM simulated discharge on the river network
Pick a completed monthly file and an hour. Values are decoded using the source CF scaling. These are simulations, not observed validation targets. Only one time slice is loaded.

In [ ]:
NWM_FILE_INDEX = -1
NWM_TIME_INDEX = 0
if files['NWM'] and not reaches.empty:
    with xr.open_dataset(files['NWM'][NWM_FILE_INDEX]) as ds:
        print(files['NWM'][NWM_FILE_INDEX].name, str(ds.time.values[NWM_TIME_INDEX]))
        q=ds.streamflow.isel(time=NWM_TIME_INDEX).load()
        values=pd.DataFrame({'comid':ds.feature_id.values.astype('int64'),'q_m3s':q.values})
    plotted=reaches.merge(values,on='comid',how='inner')
    plotted=plotted[np.isfinite(plotted.q_m3s)&(plotted.q_m3s>=0)].copy()
    if not plotted.empty:
        plotted['log_q']=np.log10(1+plotted.q_m3s)
        ax=plotted.plot(column='log_q',cmap='viridis',linewidth=0.5,legend=True,figsize=(12,7),legend_kwds={'label':'log10(1 + discharge [m³/s])'})
        if not boundary.empty:boundary.boundary.plot(ax=ax,color='black',linewidth=1)
        ax.set_title('NWM v3.0 simulated discharge');ax.set_xlabel('Longitude');ax.set_ylabel('Latitude');plt.show()
    else: print('No finite, nonnegative discharge at this time.')
else: print('No completed NWM files or reach inventory yet.')


## AORC native grid
Displays one hour from one variable file. Rainfall remains at its native hourly support; no catchment averaging occurs here.

In [ ]:
AORC_VARIABLE = 'APCP_surface'
AORC_TIME_INDEX = 0
choices=[p for p in files['AORC'] if AORC_VARIABLE in p.name]
if choices:
    with xr.open_dataset(choices[-1]) as ds:
        field=ds[AORC_VARIABLE].isel(time=AORC_TIME_INDEX).load()
        print(choices[-1].name,field.attrs)
        fig,ax=plt.subplots(figsize=(12,7));field.plot(ax=ax,cmap='viridis')
        if not boundary.empty:boundary.boundary.plot(ax=ax,color='black',linewidth=1)
        ax.set_title(f'AORC {AORC_VARIABLE}: {str(ds.time.values[AORC_TIME_INDEX])}');plt.show()
else: print('No completed AORC files for',AORC_VARIABLE,'yet. Rerun the inventory cell after downloads advance.')


## CAMELS-H observation sample
Choose a downloaded USGS ID. Missing observations are preserved; no interpolation is applied.

In [ ]:
USGS_ID = None  # e.g. '03439000'; None selects the first downloaded gauge
available=files['CAMELS-H']
path=ROOT/'raw/camelsh/flow'/f'{USGS_ID}_hourly.nc' if USGS_ID else (available[0] if available else None)
if path and path.exists():
    with xr.open_dataset(path) as ds:
        print(path.name,ds.streamflow.attrs)
        ds.streamflow.sel(time=slice('2018-01-01','2018-12-31')).plot(figsize=(12,3))
        plt.title('CAMELS-H observed discharge, 2018');plt.show()
else: print('No selected CAMELS-H flow file.')


## TVA rainfall and discharge availability
Scans all 43 files from the supplied ZIP or extracted directory. Annual counts are finite six-hour observation bins, or forecast issue times with at least one finite lead. They are not counts of matched verification samples. Historical coordinates extending into the 1800s are not treated as valid coverage.

In [ ]:
if TVA_PATH.exists():
    import netCDF4
    availability=[]
    for sid in tva.tva_nwsid.astype(str):
        for group,coord in [('observed','time'),('forecast','forecast_time')]:
            with netCDF4.Dataset('memory',memory=tva_read(f'{group}/{sid}.nc')) as ds:
                t=ds.variables[coord]
                dates=netCDF4.num2date(t[:],t.units,calendar=getattr(t,'calendar','standard'))
                years=np.array([d.year for d in dates])
                for variable in ['river_flow','precipitation']:
                    if variable in ds.variables:
                        a=np.ma.filled(ds.variables[variable][:],np.nan)
                        finite=np.isfinite(a)
                        if finite.ndim==2:finite=finite.any(axis=1)
                    else:finite=np.zeros(len(years),dtype=bool)
                    for year in range(2016,2026):
                        availability.append({'basin':sid,'dataset':group,'variable':variable,'year':year,'valid_bins_or_issues':int((finite&(years==year)).sum())})
    availability=pd.DataFrame(availability)
    fig,axes=plt.subplots(2,2,figsize=(13,15),layout='constrained')
    for ax,(group,var) in zip(axes.flat,[(g,v) for g in ['observed','forecast'] for v in ['river_flow','precipitation']]):
        tab=availability.query('dataset == @group and variable == @var').pivot(index='basin',columns='year',values='valid_bins_or_issues')
        im=ax.imshow(tab.values,aspect='auto',cmap='viridis')
        ax.set_yticks(range(len(tab)),tab.index,fontsize=7);ax.set_xticks(range(len(tab.columns)),tab.columns,rotation=45)
        ax.set_title(f'{group}: {var}');fig.colorbar(im,ax=ax,label='Finite bins / issue times')
    plt.show()
else: print('Copy TVA_headwater.zip to TVA_PATH or change the configuration.')


## Interpretation limits
- NHD connectivity is downloaded, but the final NWM graph topology still requires validation.
- A missing local gauge file does not establish that the gauge has no observations elsewhere.
- TVA observations are already six-hour sums/means; do not aggregate them again.
- TVA forecast discharge is labeled adjusted flow. Preserve actual issue times and confirm timezone and interval labels before scoring.
- No future observed rainfall or verification-time discharge is introduced as a forecast input by this notebook.
- Regional downloads include evaluation dates; training and normalization must still use the agreed training period only.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm
from IPython.display import display

ROOT = Path("/data/tva_data")
START = pd.Timestamp("1979-02-01")
STOP = pd.Timestamp("2023-02-01")  # exclusive
FULL_SCAN = False  # False: metadata checks only; True: read all values

AORC_VARIABLES = [
    "APCP_surface", "TMP_2maboveground", "SPFH_2maboveground",
    "PRES_surface", "DSWRF_surface", "DLWRF_surface",
    "UGRD_10maboveground", "VGRD_10maboveground",
]

# Build the complete expected file inventory.
expected = []
for first in pd.date_range(START, STOP, freq="MS", inclusive="left"):
    last = min(first + pd.offsets.MonthBegin(1), STOP)
    stamp = f"{first:%Y%m%dT%H}_{last:%Y%m%dT%H}"
    expected.append((
        "NWM", "streamflow",
        ROOT / "raw/nwm" / f"tennessee_nwm_{stamp}.nc",
        first, last,
    ))
    for variable in AORC_VARIABLES:
        expected.append((
            "AORC", variable,
            ROOT / "raw/aorc" / str(first.year)
            / f"tennessee_aorc_{variable}_{stamp}.nc",
            first, last,
        ))

results = []
reference_coordinates = {}

for dataset, variable, path, first, last in tqdm(
    expected, desc="Verifying monthly files"
):
    row = {
        "dataset": dataset, "variable": variable,
        "month": first.strftime("%Y-%m"),
        "file": str(path), "status": "OK",
        "missing_values": 0, "minimum": np.nan, "maximum": np.nan,
        "details": "",
    }
    issues = []

    try:
        if not path.is_file():
            raise FileNotFoundError("Expected file missing")

        marker = path.with_suffix(".complete.json")
        if not marker.is_file():
            issues.append("Completion marker missing")
        else:
            info = json.loads(marker.read_text())
            if path.stat().st_size != info["file_size_bytes"]:
                issues.append("File size differs from completion marker")
            selection = info["selection"]
            if (
                selection["start"] != str(first)
                or selection["stop_exclusive"] != str(last)
            ):
                issues.append("Completion marker dates differ")

        with xr.open_dataset(path, engine="netcdf4", chunks=None) as ds:
            if variable not in ds:
                raise ValueError(f"Variable {variable!r} missing")

            target_time = pd.date_range(
                first, last, freq="h", inclusive="left"
            )
            actual_time = pd.DatetimeIndex(ds.time.values)
            if not actual_time.equals(target_time):
                issues.append("Hourly timestamps missing, duplicated, or misaligned")

            coordinate_names = (
                ["latitude", "longitude"] if dataset == "AORC"
                else ["feature_id"]
            )
            coordinates = {}
            for name in coordinate_names:
                values = ds[name].values
                coordinates[name] = values.copy()
                if values.size == 0 or not np.isfinite(values).all():
                    issues.append(f"Invalid {name} coordinates")
                if np.unique(values).size != values.size:
                    issues.append(f"Duplicate {name} coordinates")

            if dataset not in reference_coordinates:
                reference_coordinates[dataset] = coordinates
            else:
                for name, values in coordinates.items():
                    if not np.array_equal(
                        values, reference_coordinates[dataset][name]
                    ):
                        issues.append(f"{name} ordering/grid differs between files")

            required_dims = {"time", *coordinate_names}
            if set(ds[variable].dims) != required_dims:
                issues.append(f"Unexpected dimensions: {ds[variable].dims}")

            if FULL_SCAN:
                minimum, maximum = np.inf, -np.inf
                # Read bounded blocks rather than loading a whole month.
                for offset in range(0, ds.sizes["time"], 24):
                    values = ds[variable].isel(
                        time=slice(offset, offset + 24)
                    ).values
                    finite = np.isfinite(values)
                    row["missing_values"] += int(values.size - finite.sum())
                    if finite.any():
                        minimum = min(minimum, float(values[finite].min()))
                        maximum = max(maximum, float(values[finite].max()))

                if np.isfinite(minimum):
                    row["minimum"], row["maximum"] = minimum, maximum
                else:
                    issues.append("No finite data values")

                if row["missing_values"]:
                    issues.append("Missing/nonfinite values: inspect before training")
                if variable in {"streamflow", "APCP_surface"} and minimum < 0:
                    issues.append("Negative discharge or precipitation")

    except Exception as exc:
        row["status"] = "FAIL"
        issues.append(f"{type(exc).__name__}: {exc}")

    if issues and row["status"] != "FAIL":
        row["status"] = "REVIEW"
    row["details"] = "; ".join(issues)
    results.append(row)

audit = pd.DataFrame(results)
display(
    audit.groupby(["dataset", "status"])
    .size().unstack(fill_value=0)
)

problems = audit.loc[audit.status != "OK"]
print(f"Expected files: {len(audit):,}")
print(f"Files needing attention: {len(problems):,}")
display(problems)

partials = sorted((ROOT / "raw").rglob("*.nc.part"))
print(f"Unfinished temporary files: {len(partials)}")
for path in partials[:20]:
    print(path)

output = ROOT / "metadata/monthly_timeseries_audit.csv"
output.parent.mkdir(parents=True, exist_ok=True)
audit.to_csv(output, index=False)
print(f"Audit saved: {output}")

In [ ]:
import netCDF4
from tqdm.auto import tqdm

records = []

def inspect_series(ds, dataset, station, variable, time_name):
    row = dict(
        dataset=dataset, station=str(station), variable=variable,
        status="OK", units="", first_valid=None, last_valid=None,
        valid_values=0, missing_pct=np.nan, negative_values=0,
        duplicate_times=0, nonincreasing_steps=0,
        min_step_hours=np.nan, max_step_hours=np.nan, details=""
    )
    try:
        if variable not in ds.variables:
            row.update(status="NO_VARIABLE", details="Variable absent")
            return row

        v, t = ds.variables[variable], ds.variables[time_name]
        row["units"] = getattr(v, "units", "UNSPECIFIED")

        # Decode scaling and preserve missing values.
        values = np.ma.asarray(v[:], dtype=np.float64).filled(np.nan)
        finite = np.isfinite(values)
        row["valid_values"] = int(finite.sum())
        row["missing_pct"] = (
            round(100 * (1 - finite.mean()), 2) if finite.size else np.nan
        )
        row["negative_values"] = int(((values < 0) & finite).sum())

        if len(t.dimensions) != 1:
            raise ValueError("Expected a one-dimensional time coordinate")
        time_dim = t.dimensions[0]
        time_axis = v.dimensions.index(time_dim)
        finite_by_time = np.moveaxis(finite, time_axis, 0)
        finite_by_time = finite_by_time.reshape(len(t), -1).any(axis=1)

        raw_time = np.ma.asarray(t[:], dtype=np.float64).filled(np.nan)
        if not np.isfinite(raw_time).all():
            raise ValueError("Missing/nonfinite timestamps")

        row["duplicate_times"] = int(len(raw_time) - len(np.unique(raw_time)))
        steps = np.diff(raw_time)
        row["nonincreasing_steps"] = int((steps <= 0).sum())

        calendar = getattr(t, "calendar", "standard")
        if len(raw_time) > 1:
            dates = netCDF4.num2date(
                raw_time[:2], t.units, calendar=calendar
            )
            if steps[0] != 0:
                hours_per_unit = (
                    (dates[1] - dates[0]).total_seconds() / 3600 / steps[0]
                )
                row["min_step_hours"] = float(steps.min() * hours_per_unit)
                row["max_step_hours"] = float(steps.max() * hours_per_unit)

        valid_indices = np.flatnonzero(finite_by_time)
        if valid_indices.size:
            endpoints = netCDF4.num2date(
                raw_time[valid_indices[[0, -1]]], t.units, calendar=calendar
            )
            row["first_valid"], row["last_valid"] = map(str, endpoints)
        else:
            row["status"] = "NO_VALID_DATA"

        notes = []
        if row["duplicate_times"] or row["nonincreasing_steps"]:
            notes.append("Timestamp ordering/duplicates")
        if row["negative_values"]:
            notes.append("Negative values: inspect meaning and quality flags")
        if row["units"] == "UNSPECIFIED":
            notes.append("Units unspecified")
        if notes:
            row["status"] = "REVIEW"
            row["details"] = "; ".join(notes)

    except Exception as exc:
        row.update(status="ERROR", details=f"{type(exc).__name__}: {exc}")
    return row


# CAMELS-H: compare the regional inventory against files actually present.
for station in tqdm(gauges["STAID"].astype(str).str.zfill(8).unique(),
                    desc="CAMELS-H gauges"):
    path = ROOT / "raw/camelsh/flow" / f"{station}_hourly.nc"
    if not path.exists():
        records.append(dict(
            dataset="CAMELS-H", station=station, variable="streamflow",
            status="NO_LOCAL_FILE",
            details="Check hourly-release eligibility; not automatically a download failure"
        ))
        continue
    try:
        with netCDF4.Dataset(path) as ds:
            records.append(inspect_series(
                ds, "CAMELS-H", station, "streamflow", "time"
            ))
    except Exception as exc:
        records.append(dict(
            dataset="CAMELS-H", station=station, variable="streamflow",
            status="ERROR", details=str(exc)
        ))

# TVA: audit all downloaded outlets, including those outside our study domain.
for station in tqdm(tva["tva_nwsid"].astype(str).unique(), desc="TVA outlets"):
    for group, time_name in [
        ("observed", "time"), ("forecast", "forecast_time")
    ]:
        try:
            with netCDF4.Dataset(
                "memory", memory=tva_read(f"{group}/{station}.nc")
            ) as ds:
                for variable in ["river_flow", "precipitation"]:
                    records.append(inspect_series(
                        ds, f"TVA {group}", station, variable, time_name
                    ))
        except Exception as exc:
            records.append(dict(
                dataset=f"TVA {group}", station=station, variable="FILE",
                status="ERROR", details=str(exc)
            ))

observation_audit = pd.DataFrame(records)

display(pd.crosstab(observation_audit["dataset"], observation_audit["status"]))
display(observation_audit)

output = ROOT / "metadata/observation_forecast_audit.csv"
output.parent.mkdir(parents=True, exist_ok=True)
observation_audit.to_csv(output, index=False)
print("Saved:", output)
print("Counts above represent variable records, not necessarily distinct files.")
print("First/last valid forecast dates refer to issue times.")

In [ ]:
import netCDF4
from tqdm.auto import tqdm

EXCLUDED_TVA = {"BRRT1", "PORT1S", "CDZK2", "EMGT1", "CAFT1", "MCGT1"}
TEST_START = pd.Timestamp("2020-01-01")
TEST_STOP = pd.Timestamp("2026-01-01")  # exclusive
Q_LEADS = np.arange(6, 241, 6)
RAIN_LEADS = np.arange(6, 121, 6)

matched_rows, match_errors = [], []

def read_dates(variable):
    units = variable.units.strip().lower()
    if units not in {
        "hours since 1970-01-01",
        "hours since 1970-01-01 00:00:00",
    }:
        raise ValueError(f"Unexpected timestamp units: {variable.units}")

    values = np.asarray(variable[:], dtype=float)
    return pd.to_datetime(
        values.ravel(), unit="h", origin="1970-01-01"
    ).to_numpy().reshape(values.shape)

def read_values(variable):
    return np.ma.asarray(variable[:], dtype=float).filled(np.nan)

stations = tva.loc[
    ~tva.tva_nwsid.astype(str).isin(EXCLUDED_TVA), "tva_nwsid"
].astype(str).unique()

for station in tqdm(stations, desc="Matching TVA test samples"):
    try:
        with netCDF4.Dataset(
            "memory", memory=tva_read(f"observed/{station}.nc")
        ) as obs:
            obs_times = pd.DatetimeIndex(read_dates(obs.variables["time"]))
            if obs_times.has_duplicates:
                raise ValueError("Duplicate observation timestamps")

            if "river_flow" in obs.variables:
                observed_q = read_values(obs.variables["river_flow"])
                observed_q[observed_q < 0] = np.nan
                if "river_flow_valid_hours" in obs.variables:
                    complete = read_values(
                        obs.variables["river_flow_valid_hours"]
                    ) == 6
                    observed_q[~complete] = np.nan
            else:
                observed_q = np.full(len(obs_times), np.nan)

            observed = pd.Series(observed_q, index=obs_times)

        with netCDF4.Dataset(
            "memory", memory=tva_read(f"forecast/{station}.nc")
        ) as fc:
            issue_times = pd.DatetimeIndex(
                read_dates(fc.variables["forecast_time"])
            )
            if issue_times.has_duplicates:
                raise ValueError("Duplicate forecast issue times")

            leads = read_values(fc.variables["lead_time"])
            if fc.variables["lead_time"].units.lower() != "hours":
                raise ValueError("Forecast leads are not expressed in hours")
            if len(np.unique(leads)) != len(leads):
                raise ValueError("Duplicate forecast leads")

            selected = (issue_times >= TEST_START) & (issue_times < TEST_STOP)
            issues = issue_times[selected]

            rainfall = read_values(fc.variables["precipitation"])[selected]
            forecast_q = (
                read_values(fc.variables["river_flow"])[selected]
                if "river_flow" in fc.variables
                else np.full(rainfall.shape, np.nan)
            )

            # Check recorded valid times against issue + lead.
            if "valid_time" in fc.variables:
                recorded = np.asarray(
                    read_dates(fc.variables["valid_time"])
                )[selected]
                calculated = (
                    issues.to_numpy()[:, None]
                    + pd.to_timedelta(leads, unit="h").to_numpy()[None, :]
                )
                if not np.array_equal(recorded, calculated):
                    raise ValueError("valid_time differs from issue + lead")

            rain_indices = [
                np.flatnonzero(leads == lead) for lead in RAIN_LEADS
            ]
            if any(len(index) != 1 for index in rain_indices):
                raise ValueError("Required rainfall leads are missing")

            rain = rainfall[:, [index[0] for index in rain_indices]]
            rain_ready = (np.isfinite(rain) & (rain >= 0)).all(axis=1)

            for lead in Q_LEADS:
                indices = np.flatnonzero(leads == lead)
                if len(indices) != 1:
                    raise ValueError(f"Missing discharge lead: {lead}")

                q = forecast_q[:, indices[0]]
                valid_times = issues + pd.Timedelta(hours=int(lead))
                targets = observed.reindex(valid_times).to_numpy()

                # Keep every target interval within the reserved test period.
                within_test = (
                    (valid_times >= TEST_START)
                    & (valid_times + pd.Timedelta(hours=6) <= TEST_STOP)
                )
                observed_ready = np.isfinite(targets) & within_test
                tva_ready = np.isfinite(q) & (q >= 0)

                for year in range(2020, 2026):
                    year_mask = issues.year == year
                    matched_rows.append({
                        "station": station,
                        "issue_year": year,
                        "lead_hours": int(lead),
                        "stored_issues": int(year_mask.sum()),
                        "five_day_rain_ready": int(
                            (year_mask & rain_ready).sum()
                        ),
                        "rain_and_observation_pairs": int(
                            (year_mask & rain_ready & observed_ready).sum()
                        ),
                        "rain_observation_TVA_pairs": int(
                            (year_mask & rain_ready
                             & observed_ready & tva_ready).sum()
                        ),
                    })

    except Exception as exc:
        match_errors.append({"station": station, "error": str(exc)})

matched = pd.DataFrame(matched_rows)
errors = pd.DataFrame(match_errors)

if not matched.empty:
    print("Candidate comparison pairs at selected leads:")
    display(
        matched.loc[matched.lead_hours.isin([6, 24, 120, 240])]
        .pivot_table(
            index=["station", "lead_hours"],
            columns="issue_year",
            values="rain_observation_TVA_pairs",
            aggfunc="sum"
        )
    )
    print("Outlets with at least one candidate pair, by year and lead:")
    display(
        matched.assign(
            eligible=matched.rain_observation_TVA_pairs > 0
        ).groupby(["lead_hours", "issue_year"])
        .eligible.sum().unstack()
    )

print("Errors:")
display(errors)

(ROOT / "metadata").mkdir(parents=True, exist_ok=True)
matched.to_csv(ROOT / "metadata/TVA_test_matched_coverage.csv", index=False)
errors.to_csv(ROOT / "metadata/TVA_test_matching_errors.csv", index=False)

In [ ]:
from collections import deque
import numpy as np
import pandas as pd
import xarray as xr

inventory = pd.read_csv(ROOT / "metadata/tennessee_reach_inventory.csv")
connections = pd.read_parquet(
    ROOT / "metadata/tennessee_nhd_connectivity.parquet"
)

def as_ids(values, name):
    numeric = pd.to_numeric(values, errors="raise")
    if numeric.isna().any() or not np.isfinite(numeric).all():
        raise ValueError(f"{name}: missing/nonfinite IDs")
    if (numeric != np.floor(numeric)).any():
        raise ValueError(f"{name}: noninteger IDs")
    return numeric.astype("int64")

def as_bool(values):
    mapped = values.astype(str).str.lower().map({
        "true": True, "false": False, "1": True, "0": False
    })
    if mapped.isna().any():
        raise ValueError("Unexpected availability flag values")
    return mapped.astype(bool)

inventory["comid"] = as_ids(inventory.comid, "COMID")
assert inventory.comid.is_unique, "Duplicate inventory COMIDs"
assert (inventory.comid > 0).all(), "Invalid inventory COMIDs"

inventory["in_nwm_v30"] = as_bool(inventory.in_nwm_v30)
inventory["has_catchment"] = as_bool(inventory.has_catchment)

for column in ["fromcomid", "tocomid"]:
    connections[column] = as_ids(connections[column], column)

# Use the actual discharge-file order as the candidate node order.
nwm_files = sorted((ROOT / "raw/nwm").glob("*.nc"))
assert nwm_files, "No NWM discharge files found"
with xr.open_dataset(nwm_files[0]) as ds:
    node_ids = as_ids(pd.Series(ds.feature_id.values), "NWM feature_id").to_numpy()

assert len(np.unique(node_ids)) == len(node_ids), "Duplicate NWM feature IDs"
expected_ids = set(inventory.loc[inventory.in_nwm_v30, "comid"])
assert set(node_ids) == expected_ids, "NWM IDs differ from matched inventory"

nodes = (
    inventory.set_index("comid").loc[node_ids].reset_index()
)
nodes.insert(0, "node_index", np.arange(len(nodes)))
node_index = pd.Series(nodes.node_index.values, index=nodes.comid)
all_ids = set(inventory.comid)

# Zero IDs denote boundary/headwater entries, not graph nodes.
positive = connections.loc[
    (connections.fromcomid > 0) & (connections.tocomid > 0)
].copy()
regional = positive.loc[
    positive.fromcomid.isin(all_ids) & positive.tocomid.isin(all_ids)
].copy()

boundary_links = positive.loc[
    positive.fromcomid.isin(all_ids) ^ positive.tocomid.isin(all_ids)
].copy()

matched_mask = (
    regional.fromcomid.isin(expected_ids)
    & regional.tocomid.isin(expected_ids)
)
excluded_links = regional.loc[~matched_mask].copy()
candidate = regional.loc[matched_mask].copy()

duplicate_edges = candidate.loc[
    candidate.duplicated(["fromcomid", "tocomid"], keep=False)
].copy()
self_loops = candidate.loc[
    candidate.fromcomid == candidate.tocomid
].copy()

# Deduplicate only for diagnostics; keep original records above.
edges = candidate[["fromcomid", "tocomid"]].drop_duplicates()
edges = edges.loc[edges.fromcomid != edges.tocomid].copy()
edges["source"] = edges.fromcomid.map(node_index).astype("int64")
edges["target"] = edges.tocomid.map(node_index).astype("int64")

n = len(nodes)
src, dst = edges.source.to_numpy(), edges.target.to_numpy()
indegree = np.bincount(dst, minlength=n)
outdegree = np.bincount(src, minlength=n)

# Kahn's algorithm: test whether the candidate directed graph is acyclic.
adjacency = [[] for _ in range(n)]
for source, target in zip(src, dst):
    adjacency[source].append(target)

remaining_degree = indegree.copy()
queue = deque(np.flatnonzero(remaining_degree == 0))
visited = 0
while queue:
    source = queue.popleft()
    visited += 1
    for target in adjacency[source]:
        remaining_degree[target] -= 1
        if remaining_degree[target] == 0:
            queue.append(target)

# Weakly connected components using union-find.
parent = np.arange(n)
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

for source, target in zip(src, dst):
    a, b = find(source), find(target)
    if a != b:
        parent[b] = a

nodes["component"] = [find(i) for i in range(n)]
nodes["in_degree"] = indegree
nodes["out_degree"] = outdegree
component_sizes = nodes.groupby("component").size().sort_values(ascending=False)

# NHDPlus HydroSeq should decrease downstream where valid.
hydroseq = pd.to_numeric(
    inventory.set_index("comid")["hydroseq"], errors="coerce"
)
up = edges.fromcomid.map(hydroseq)
down = edges.tocomid.map(hydroseq)
known = up.notna() & down.notna() & (up > 0) & (down > 0)
direction_review = edges.loc[known & (up <= down)].copy()

summary = pd.Series({
    "NHD inventory reaches": len(inventory),
    "Matched NWM nodes": n,
    "Unmatched NHD reaches": int((~inventory.in_nwm_v30).sum()),
    "Matched nodes without catchments": int((~nodes.has_catchment).sum()),
    "Candidate directed edges": len(edges),
    "Duplicate edge records": len(duplicate_edges),
    "Self-loop records": len(self_loops),
    "Regional edges involving unmatched reaches": len(excluded_links),
    "Links crossing inventory boundary": len(boundary_links),
    "Weakly connected components": len(component_sizes),
    "Largest component nodes": int(component_sizes.iloc[0]),
    "Isolated nodes": int(((indegree == 0) & (outdegree == 0)).sum()),
    "Nodes with multiple downstream edges": int((outdegree > 1).sum()),
    "HydroSeq direction mismatches": len(direction_review),
    "Nodes unresolved by acyclicity test": n - visited,
}, name="count")

display(summary.to_frame())
print("Largest connected components:")
display(component_sizes.head(10).to_frame("nodes"))

print("Direction mismatches:")
display(direction_review.head(10))
print("Self-loops:")
display(self_loops.head(10))

print(
    "Candidate graph is acyclic:", visited == n,
    "\nCatchment availability uses inventory flags; polygon validation comes next."
)


In [ ]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

network = reaches.copy()
network["comid"] = pd.to_numeric(
    network["comid"], errors="raise"
).astype("int64")
network = network.to_crs(5070)
basin_map = boundary.to_crs(5070)

def finish_map(ax, title, bounds=None):
    if not basin_map.empty:
        basin_map.boundary.plot(ax=ax, color="black", linewidth=0.9)
        if bounds is None:
            bounds = basin_map.total_bounds

    if bounds is not None:
        xmin, ymin, xmax, ymax = bounds
        margin = max(xmax - xmin, ymax - ymin) * 0.025
        ax.set_xlim(xmin - margin, xmax + margin)
        ax.set_ylim(ymin - margin, ymax + margin)

    ax.set_aspect("equal")
    ax.set_axis_off()
    ax.set_title(title)
EXCLUDED_TVA = {"BRRT1", "PORT1S", "CDZK2", "EMGT1", "CAFT1", "MCGT1"}
OUTLET_ID = "BIGT1"
OUTLET_RADIUS_KM = 8

selected = tva.loc[
    ~tva.tva_nwsid.astype(str).isin(EXCLUDED_TVA)
].copy()

def location_points(frame, lon_column, lat_column):
    frame = frame.copy()
    frame[lon_column] = pd.to_numeric(frame[lon_column], errors="coerce")
    frame[lat_column] = pd.to_numeric(frame[lat_column], errors="coerce")
    valid = (
        frame[lon_column].between(-180, 180)
        & frame[lat_column].between(-90, 90)
    )
    frame = frame.loc[valid]
    return gpd.GeoDataFrame(
        frame,
        geometry=gpd.points_from_xy(frame[lon_column], frame[lat_column]),
        crs=4326
    ).to_crs(5070)

outlets = location_points(
    selected, "tva_station_longitude", "tva_station_latitude"
)
paired = location_points(
    selected, "camelsh_station_longitude", "camelsh_station_latitude"
)

fig, ax = plt.subplots(figsize=(18, 11), dpi=150)
network.plot(ax=ax, color="#91bbd6", linewidth=0.35)
if not paired.empty:
    paired.plot(ax=ax, facecolor="none", edgecolor="#198b45",
                markersize=100, linewidth=1.5, zorder=4)
if not outlets.empty:
    outlets.plot(ax=ax, marker="^", color="#9225a5",
                 markersize=35, zorder=5)
    for row in outlets.itertuples():
        ax.annotate(str(row.tva_nwsid), (row.geometry.x, row.geometry.y),
                    xytext=(4, 4), textcoords="offset points", fontsize=7)

finish_map(ax, "Selected TVA outlets and CSV-paired CAMELS-H locations")
ax.legend(handles=[
    Line2D([0], [0], marker="^", color="none",
           markerfacecolor="#9225a5", label="TVA outlet"),
    Line2D([0], [0], marker="o", color="none",
           markeredgecolor="#198b45", markerfacecolor="none",
           label="CSV-paired CAMELS-H location")
], loc="lower left")
plt.tight_layout()
plt.show()

# Read only catchments intersecting the local outlet window.
match = outlets.loc[outlets.tva_nwsid.astype(str) == OUTLET_ID]
if match.empty:
    raise ValueError(f"No usable TVA coordinates for {OUTLET_ID}")
point = match.geometry.iloc[0]
radius = OUTLET_RADIUS_KM * 1000
bounds = (point.x-radius, point.y-radius, point.x+radius, point.y+radius)

manifest = json.loads((ROOT / "metadata/nhdplus_download.json").read_text())
component = next(
    c for c in manifest["components"]
    if c["component"] == "NHDPlusCatchment"
)
paths = list(Path(component["extracted_directory"]).rglob("Catchment.shp"))
assert len(paths) == 1, "Missing or ambiguous Catchment.shp"

import pyogrio
from pyproj import Transformer

info = pyogrio.read_info(paths[0])
source_bbox = Transformer.from_crs(
    5070, info["crs"], always_xy=True
).transform_bounds(*bounds)
catchments = gpd.read_file(paths[0], bbox=source_bbox).to_crs(5070)

fig, ax = plt.subplots(figsize=(12, 10), dpi=150)
if not catchments.empty:
    catchments.plot(ax=ax, facecolor="#d7ead2", edgecolor="#56834a",
                    linewidth=0.5, alpha=0.5)
local_network = network.cx[bounds[0]:bounds[2], bounds[1]:bounds[3]]
local_network.plot(ax=ax, color="#2474ac", linewidth=0.9)

candidate_comid = pd.to_numeric(
    match["camelsh_outlet_comid"], errors="coerce"
).iloc[0]
if pd.notna(candidate_comid):
    candidate = network.loc[network.comid == int(candidate_comid)]
    if not candidate.empty:
        candidate.plot(ax=ax, color="#f08020", linewidth=3, zorder=4)

local_pair = paired.loc[paired.tva_nwsid.astype(str) == OUTLET_ID]
if not local_pair.empty:
    local_pair.plot(ax=ax, facecolor="none", edgecolor="#198b45",
                    markersize=180, linewidth=2, zorder=5)
match.plot(ax=ax, marker="^", color="#9225a5", markersize=90, zorder=6)

finish_map(ax, f"{OUTLET_ID} — catchments and candidate outlet reach", bounds)
ax.legend(handles=[
    Line2D([0], [0], color="#2474ac", lw=2, label="River reaches"),
    Line2D([0], [0], color="#f08020", lw=3,
           label="CSV candidate CAMELS-H outlet reach"),
    Line2D([0], [0], marker="^", color="none",
           markerfacecolor="#9225a5", label="TVA outlet"),
    Line2D([0], [0], marker="o", color="none",
           markeredgecolor="#198b45", markerfacecolor="none",
           label="CSV-paired CAMELS-H location")
], loc="lower left")
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm
from IPython.display import display

ROOT = Path("/data/tva_data")
VARIABLES = [
    "APCP_surface", "TMP_2maboveground", "SPFH_2maboveground",
    "PRES_surface", "DSWRF_surface", "DLWRF_surface",
    "UGRD_10maboveground", "VGRD_10maboveground", "streamflow",
]

nodes = pd.read_parquet(
    ROOT / "processed/graph_candidate/nodes.parquet"
).sort_values("node_index")

expected_ids = nodes.comid.to_numpy(dtype="int64")
expected_hash = hashlib.sha256(
    expected_ids.astype("<i8").tobytes()
).hexdigest()

file_rows, coverage_rows = [], []

months = pd.date_range("1979-02-01", "2023-02-01",
                       freq="MS", inclusive="left")

for first in tqdm(months, desc="Auditing monthly outputs"):
    last = first + pd.offsets.MonthBegin(1)
    stamp = f"{first:%Y%m%dT%H}_{last:%Y%m%dT%H}"
    path = (
        ROOT / "processed/six_hour_nodes" / str(first.year)
        / f"tennessee_{stamp}_six_hour_nodes.nc"
    )
    marker = path.with_suffix(".complete.json")
    expected_time = pd.date_range(first, last, freq="6h",
                                  inclusive="left")
    errors, notes = [], []
    row = {"month": f"{first:%Y-%m}", "file": str(path)}

    try:
        if not path.is_file() or not marker.is_file():
            raise FileNotFoundError("Output or completion marker missing")

        info = json.loads(marker.read_text())

        if info["file_size_bytes"] != path.stat().st_size:
            errors.append("File size differs from completion marker")
        if info["nodes"] != len(expected_ids):
            errors.append("Marker node count differs")
        if info["six_hour_intervals"] != len(expected_time):
            errors.append("Marker interval count differs")
        if info["fingerprint"]["node_order_sha256"] != expected_hash:
            errors.append("Marker node-order hash differs")

        # Reads only coordinates and headers.
        with xr.open_dataset(path, engine="netcdf4",
                             chunks=None, cache=False) as ds:
            if not pd.DatetimeIndex(ds.time.values).equals(expected_time):
                errors.append("Six-hour timestamps differ")
            if not np.array_equal(ds.feature_id.values, expected_ids):
                errors.append("Node IDs or ordering differ")

            for variable in VARIABLES:
                count_name = variable + "_valid_hours"
                for name in (variable, count_name):
                    if name not in ds:
                        errors.append(f"Missing variable: {name}")
                    elif ds[name].dims != ("time", "feature_id"):
                        errors.append(f"Unexpected dimensions: {name}")

            if "time_bounds" not in ds:
                errors.append("Time bounds missing")
            else:
                bounds = ds.time_bounds.values
                wanted = np.column_stack([
                    expected_time.values,
                    (expected_time + pd.Timedelta(hours=6)).values,
                ])
                if not np.array_equal(bounds, wanted):
                    errors.append("Interval bounds differ")

        entries = info.get("audit", [])
        names = [entry["variable"] for entry in entries]
        if sorted(names) != sorted(VARIABLES):
            errors.append("Coverage summary missing/duplicate variables")

        total = len(expected_time) * len(expected_ids)
        for entry in entries:
            valid = entry["valid_intervals"]
            masked = entry["masked_intervals"]
            partial = entry["partial_intervals"]

            if (
                valid < 0 or masked < 0 or partial < 0
                or valid + masked != total or partial > masked
            ):
                errors.append(f"Invalid coverage counts: {entry['variable']}")

            coverage_rows.append({
                **entry,
                "month": f"{first:%Y-%m}",
                "masked_percent": 100 * masked / total,
            })
            if partial:
                notes.append(f"{entry['variable']}: {partial:,} partial intervals")

        if not info.get("boundary_rainfall_file_present", False):
            notes.append("Following-month rainfall file unavailable")

        row["size_MiB"] = path.stat().st_size / 2**20

    except Exception as exc:
        errors.append(f"{type(exc).__name__}: {exc}")

    row.update(
        status="FAIL" if errors else ("REVIEW" if notes else "OK"),
        errors="; ".join(errors),
        notes="; ".join(notes),
    )
    file_rows.append(row)

audit = pd.DataFrame(file_rows)
coverage = pd.DataFrame(coverage_rows)

display(audit.groupby("status").size().rename("months").to_frame())
display(audit.loc[audit.status != "OK",
                  ["month", "status", "errors", "notes"]])

if not coverage.empty:
    display(
        coverage.groupby("variable")[
            ["valid_intervals", "masked_intervals", "partial_intervals"]
        ].sum()
    )

print(f"Expected months: {len(months)}")
print(f"Failed months: {(audit.status == 'FAIL').sum()}")
print(f"Output size: {audit.get('size_MiB', pd.Series(dtype=float)).sum()/1024:.2f} GiB")

metadata = ROOT / "metadata"
metadata.mkdir(parents=True, exist_ok=True)
audit.to_csv(metadata / "six_hour_file_audit.csv", index=False)
coverage.to_csv(metadata / "six_hour_coverage_audit.csv", index=False)

print("Audit reports saved under /data/tva_data/metadata/")